In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

### ※ SELECT文の実行順序

SQL文は、記述されている順番ではなく、次の順番で処理される。

```text
FROM → WHERE → SELECT → ORDER BY → LIMIT
```

#### 1. FROM句

最初に、`actor`テーブルのデータを取得する。

In [4]:
%%sql

SELECT *
FROM actor

LIMIT 5;

actor_id,first_name,last_name,last_update
1,PENELOPE,GUINESS,2006-02-15 04:34:33
2,NICK,WAHLBERG,2006-02-15 04:34:33
3,ED,CHASE,2006-02-15 04:34:33
4,JENNIFER,DAVIS,2006-02-15 04:34:33
5,JOHNNY,LOLLOBRIGIDA,2006-02-15 04:34:33


#### 2. WHERE句

`actor_id`が偶数のデータだけに絞り込む。

In [5]:
%%sql

SELECT *
FROM actor
WHERE actor_id % 2 = 0

LIMIT 5;

actor_id,first_name,last_name,last_update
2,NICK,WAHLBERG,2006-02-15 04:34:33
4,JENNIFER,DAVIS,2006-02-15 04:34:33
6,BETTE,NICHOLSON,2006-02-15 04:34:33
8,MATTHEW,JOHANSSON,2006-02-15 04:34:33
10,CHRISTIAN,GABLE,2006-02-15 04:34:33


#### 3. SELECT句

必要な`actor_id`と`first_name`だけを取得する。

In [6]:
%%sql

SELECT
    actor_id,
    first_name
FROM actor
WHERE actor_id % 2 = 0

LIMIT 5;

actor_id,first_name
2,NICK
4,JENNIFER
6,BETTE
8,MATTHEW
10,CHRISTIAN


#### 4. ORDER BY句

`actor_id`の降順に並べ替える。

In [7]:
%%sql

SELECT
    actor_id,
    first_name
FROM actor
WHERE actor_id % 2 = 0
ORDER BY actor_id DESC

LIMIT 5;

actor_id,first_name
200,THORA
198,MARY
196,BELA
194,MERYL
192,JOHN


#### 5. LIMIT句

並べ替えた結果から、先頭の特定の数を取得する。

In [9]:
%%sql

SELECT
    actor_id,
    first_name
FROM actor
WHERE actor_id % 2 = 0
ORDER BY actor_id DESC
LIMIT 3;

actor_id,first_name
200,THORA
198,MARY
196,BELA


## 1. GROUP BY

`GROUP BY`は、`FROM句`や`WHERE句`で取得した結果を、指定したカラムの値が同じ行ごとに1つのグループにまとめる機能である。

作成したグループに対して、平均・合計・件数などの集計関数（Aggregate Function）を使用する。

In [10]:
%%sql

CREATE TABLE person(
	id int primary key,
	name varchar(50),
	gender char(1)
);

++
||
++
++

In [11]:
%%sql

INSERT INTO person (id, name, gender) VALUES
(1,  'ハウル',   'M'),
(2,  '千尋',     'F'),
(3,  'パズー',   'M'),
(4,  'シータ',   'F'),
(5,  'アシタカ', 'M'),
(6,  'サン',     'F'),
(7,  'ポルコ',   'M'),
(8,  'キキ',     'F'),
(9,  'ハク',     'M'),
(10, 'ソフィー', 'F');

++
||
++
++

##### 1) FROM句の結果セット

In [16]:
%%sql

SELECT * 
FROM person              

id,name,gender
1,ハウル,M
2,千尋,F
3,パズー,M
4,シータ,F
5,アシタカ,M
6,サン,F
7,ポルコ,M
8,キキ,F
9,ハク,M
10,ソフィー,F


##### 2) GROUP BYで作成されるグループ（F・Mの2グループ）

In [17]:
%%sql

SELECT *
FROM person
WHERE gender = 'F';

id,name,gender
2,千尋,F
4,シータ,F
6,サン,F
8,キキ,F
10,ソフィー,F


In [18]:
%%sql

SELECT *
FROM person
WHERE gender = 'M';

id,name,gender
1,ハウル,M
3,パズー,M
5,アシタカ,M
7,ポルコ,M
9,ハク,M


##### 3) 集計関数による集計結果

In [19]:
%%sql

SELECT gender, COUNT(*)  -- 3. 各グループを集計して取得
FROM person              -- 1. 1つの結果セットを作成
GROUP BY gender;         -- 2. MとFの2つのグループを作成

gender,COUNT(*)
M,5
F,5


## 2. GROUP BYでよく使用される集計関数

`GROUP BY句`で指定したカラムと、`COUNT`、`SUM`、`AVG`、`MAX`、`MIN`などの集計関数を使用する。

#### 1) 映画の件数

In [21]:
%%sql

SELECT
    rating,
    COUNT(*) AS 映画数
FROM film
GROUP BY rating;

rating,映画数
PG,194
G,178
NC-17,210
PG-13,223
R,195


#### 2) 平均レンタル料金

In [22]:
%%sql

SELECT
    rating,
    AVG(rental_rate) AS 平均レンタル料金
FROM film
GROUP BY rating;

rating,平均レンタル料金
PG,3.051856
G,2.888876
NC-17,2.970952
PG-13,3.034843
R,2.938718


#### 3) 上映時間の合計

In [23]:
%%sql

SELECT
    rating,
    SUM(length) AS 合計上映時間
FROM film
GROUP BY rating;

rating,合計上映時間
PG,21729
G,19767
NC-17,23778
PG-13,26859
R,23139


#### 4) 最長上映時間

In [24]:
%%sql

SELECT
    rating,
    MAX(length) AS 最長上映時間
FROM film
GROUP BY rating;

rating,最長上映時間
PG,185
G,185
NC-17,184
PG-13,185
R,185


#### 5) 最短上映時間

In [25]:
%%sql

SELECT
    rating,
    MIN(length) AS 最短上映時間
FROM film
GROUP BY rating;

rating,最短上映時間
PG,46
G,47
NC-17,46
PG-13,46
R,49


#### 6) 複数の集計関数を同時に使用する

In [26]:
%%sql

SELECT
    rating,
    COUNT(*) AS 映画数,
    AVG(length) AS 平均上映時間,
    MAX(length) AS 最長上映時間,
    MIN(length) AS 最短上映時間
FROM film
GROUP BY rating;

rating,映画数,平均上映時間,最長上映時間,最短上映時間
PG,194,112.0052,185,46
G,178,111.0506,185,47
NC-17,210,113.2286,184,46
PG-13,223,120.4439,185,46
R,195,118.6615,185,49


#### ※ GROUP BY句を省略する場合

- `GROUP BY句`を指定せずに集計関数を使用すると、テーブル全体を1つのグループとして集計する。

In [27]:
%%sql

SELECT COUNT(*)
FROM film;

COUNT(*)
1000


- 概念的には、次のように`GROUP BY NULL`を指定した場合と同じである。

In [28]:
%%sql

SELECT COUNT(*)
FROM film
GROUP BY NULL;

COUNT(*)
1000


## 3. 複数のカラムを指定するGROUP BY

`FROM句`や`JOIN句`が先に処理されるため、テーブルを結合した後に`GROUP BY`が実行される。

In [33]:
%%sql

SELECT
    c.name,
    f.rating,
    COUNT(*)
FROM category AS c
JOIN film_category AS fc
    ON c.category_id = fc.category_id
JOIN film AS f
    ON fc.film_id = f.film_id
GROUP BY
    c.name,
    f.rating

LIMIT 10;

name,rating,COUNT(*)
Action,PG,9
Action,R,14
Action,NC-17,12
Action,G,18
Action,PG-13,11
Animation,PG-13,19
Animation,R,8
Animation,NC-17,15
Animation,PG,11
Animation,G,13


### 作成されるグループ

`c.name`と`f.rating`の値の組み合わせごとに、1つのグループが作成される。

```text
Action + PG
Action + R
Action + G
Animation + PG
Animation + R
...
```

その後、`COUNT(*)`によって、それぞれのグループに含まれる映画の件数を集計する。

```text
FROM・JOIN
    ↓
3つのテーブルを結合する
    ↓
GROUP BY c.name, f.rating
    ↓
カテゴリ名とレーティングの組み合わせごとにグループ化する
    ↓
COUNT(*)
    ↓
各グループの映画数を集計する
```

#### R・Gレーティングの映画だけをグループ化する

`WHERE句`で対象となる行を先に絞り込み、その後に`GROUP BY`を実行する。

In [ ]:
%%sql

SELECT
    rating,
    COUNT(*)  -- 4. 各グループのレーティングと映画数を取得する
FROM film  -- 1. filmテーブルのデータを取得する
WHERE rating IN ('G', 'R')  -- 2. GまたはRの映画だけに絞り込む
GROUP BY rating;  -- 3. GグループとRグループを作成する

rating,COUNT(*)
G,178
R,195


## 4. HAVING句

`HAVING句`は、`GROUP BY`でグループを作成した後、集計関数の結果を条件として絞り込むために使用する。

- `WHERE句`：グループ化する前に、行を絞り込む
- `HAVING句`：グループ化した後に、集計結果を絞り込む

#### 1) 映画数が200本以上のレーティングを取得する

In [34]:
%%sql

SELECT
    rating,
    COUNT(*) AS cnt
FROM film
GROUP BY rating
HAVING COUNT(*) >= 200;

rating,cnt
NC-17,210
PG-13,223


#### 2) R以外のレーティングのうち、平均上映時間が120分以上のグループを取得する

In [35]:
%%sql

SELECT
    rating,
    AVG(length) AS avg_length
FROM film
WHERE rating <> 'R'
GROUP BY rating
HAVING AVG(length) >= 120;

rating,avg_length
PG-13,120.4439


#### 3) 平均上映時間の降順に並べ替える

In [36]:
%%sql

SELECT
    rating,
    AVG(length) AS avg_length
FROM film
GROUP BY rating
ORDER BY avg_length DESC;

rating,avg_length
PG-13,120.4439
R,118.6615
NC-17,113.2286
PG,112.0052
G,111.0506
